<a href="https://colab.research.google.com/github/ImagingDataCommons/IDC-Tutorials/blob/master/notebooks/pathomics/grandqc_slide_quality_with_idc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Slide Quality Control with GrandQC and NCI Imaging Data Commons

Automated quality control (QC) is an essential preprocessing step in computational pathology pipelines. Artifacts such as tissue folds, out-of-focus regions, pen markings, and air bubbles can introduce noise that degrades the performance of downstream AI models.

[**GrandQC**](https://github.com/cpath-ukk/grandqc) is an open-source tool for automated tissue detection and multi-class artifact segmentation in whole slide images (WSIs). It was validated across slides from 19 international pathology departments and published in:

> Weng Z., Seper A., Pryalukhin A., et al. *GrandQC: A comprehensive solution to quality control problem in digital pathology.* Nature Communications 15, 10685 (2024). https://doi.org/10.1038/s41467-024-54769-y

[**NCI Imaging Data Commons (IDC)**](https://portal.imaging.datacommons.cancer.gov/) hosts ~100 TB of cancer imaging data, including thousands of H&E-stained whole slide images from TCGA, CPTAC, HTAN, and other programs — all stored as DICOM and freely accessible without authentication.

This notebook demonstrates how to:
1. **Discover** H&E-stained whole slide images in IDC using `idc-index`
2. **Set up** GrandQC and download its pre-trained models
3. **Download** slides from IDC and run GrandQC directly on DICOM files
4. **Visualize** tissue detection and artifact segmentation results
5. **Leverage** GrandQC's pre-computed QC masks for all 32 TCGA cohorts

## Disclaimer

The code and data of this repository are provided to promote reproducible research. They are not intended for clinical care or commercial use.

The software is provided "as is", without warranty of any kind, express or implied, including but not limited to the warranties of merchantability, fitness for a particular purpose and noninfringement. In no event shall the authors or copyright holders be liable for any claim, damages or other liability, whether in an action of contract, tort or otherwise, arising from, out of or in connection with the software or the use or other dealings in the software.

**GrandQC license**: Creative Commons Attribution-NonCommercial-ShareAlike 4.0 (CC BY-NC-SA 4.0). Non-commercial research use only; citation of the GrandQC paper is required.

## Part 1: Discover H&E Slides in IDC

IDC provides the `idc-index` Python package for querying slide metadata and downloading DICOM files. We start by installing it and exploring the available whole slide image collections.

In [1]:
%%capture
!pip install --upgrade idc-index openslide-python openslide-bin

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import IFrame, display
from idc_index import IDCClient

idc_client = IDCClient()
print(f"IDC data version: {idc_client.get_idc_version()}")

IDC data version: v24


### 1.1 Overview of Slide Microscopy Collections

IDC stores whole slide images using the DICOM Slide Microscopy (SM) standard. The `sm_index` table extends the main metadata index with pathology-specific attributes including staining protocol, tissue type, pixel spacing, and image dimensions.

In [3]:
# Load the slide microscopy index
idc_client.fetch_index("sm_index")

# Summary of all SM collections
sm_collections = idc_client.sql_query("""
    SELECT
        i.collection_id,
        COUNT(DISTINCT i.PatientID)      AS patients,
        COUNT(DISTINCT i.SeriesInstanceUID) AS series,
        ROUND(SUM(i.series_size_MB) / 1024.0, 1) AS size_GB,
        i.license_short_name
    FROM index i
    WHERE i.Modality = 'SM'
    GROUP BY i.collection_id, i.license_short_name
    ORDER BY patients DESC
""")

print(f"Total SM collections: {len(sm_collections)}")
print(f"Total SM series:      {sm_collections['series'].sum():,}")
print(f"Total size:           {sm_collections['size_GB'].sum():.0f} GB")
print()
sm_collections.head(15)

Total SM collections: 73
Total SM series:      76,299
Total size:           47871 GB



,collection_id,patients,series,size_GB,license_short_name
0,ccdi_mci,4407,4576,4548.0,CC BY 4.0
1,tcga_brca,1098,3111,1678.8,CC BY 3.0
2,gtex,971,25503,8354.2,CC BY 4.0
3,pdxnet,919,919,122.7,CC BY 4.0
4,tcga_gbm,607,2053,639.9,CC BY 3.0
5,tcga_ov,590,1481,476.5,CC BY 3.0
6,tcga_ucec,560,1371,1078.5,CC BY 3.0
7,tcga_kirc,537,2173,811.7,CC BY 3.0
8,tcga_hnsc,523,1263,560.5,CC BY 3.0
9,tcga_luad,522,1608,631.1,CC BY 3.0


### 1.2 Filter for H&E-Stained Slides

The `sm_index` table includes a `staining_usingSubstance_CodeMeaning` column (stored as an array) that records the staining protocol for each slide. We filter for slides stained with hematoxylin, which identifies H&E preparations.

In [4]:
# Count H&E slides per collection
he_by_collection = idc_client.sql_query("""
    SELECT
        i.collection_id,
        COUNT(DISTINCT i.PatientID)         AS patients,
        COUNT(DISTINCT i.SeriesInstanceUID) AS he_series,
        ROUND(SUM(i.series_size_MB) / 1024.0, 1) AS size_GB
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
    GROUP BY i.collection_id
    ORDER BY he_series DESC
""")

print(f"Collections with H&E slides: {len(he_by_collection)}")
print(f"Total H&E series: {he_by_collection['he_series'].sum():,}")
he_by_collection.head(15)

Collections with H&E slides: 70
Total H&E series: 73,422


,collection_id,patients,he_series,size_GB
0,gtex,971,25503,8354.2
1,ccdi_mci,4401,4569,4545.8
2,tcga_brca,1098,3111,1678.8
3,tcga_kirc,537,2173,811.7
4,tcga_gbm,607,2053,639.9
5,tcga_lusc,504,1612,610.4
6,tcga_luad,522,1608,631.1
7,tcga_lgg,516,1572,1166.6
8,tcga_ov,590,1481,476.5
9,tcga_coad,460,1442,446.5


### 1.3 Select Slides for Quality Control

For this tutorial we work with slides from the [TCGA-BRCA](https://portal.imaging.datacommons.cancer.gov/explore/filters/?collection_id=tcga_brca) collection (The Cancer Genome Atlas — Breast Cancer). This collection is a good choice because:

- It contains over 3,000 H&E-stained slides at 40× magnification
- GrandQC provides **pre-computed QC masks for all TCGA cohorts** (demonstrated in Part 6)
- Slides span both tumor and adjacent normal tissue, providing variety in tissue composition

We query the `sm_index` to retrieve per-slide metadata including the `ContainerIdentifier` (the original TCGA slide barcode), which we will later use to match IDC series against the pre-computed GrandQC masks.

In [5]:
# Query TCGA-BRCA H&E slides with key metadata
tcga_brca_he = idc_client.sql_query("""
    SELECT
        i.SeriesInstanceUID,
        i.PatientID,
        s.ContainerIdentifier,
        s.primaryAnatomicStructureModifier_CodeMeaning AS tissue_type,
        s.max_TotalPixelMatrixColumns                  AS width_px,
        s.max_TotalPixelMatrixRows                     AS height_px,
        s.min_PixelSpacing_2sf                         AS pixel_spacing_mm,
        s.ObjectiveLensPower                           AS objective_power,
        ROUND(i.series_size_MB, 1)                     AS size_MB
    FROM index i
    JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i.collection_id = 'tcga_brca'
      AND array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
    ORDER BY i.series_size_MB ASC
""")

print(f"TCGA-BRCA H&E slides: {len(tcga_brca_he)}")
print(f"Tissue types: {tcga_brca_he['tissue_type'].value_counts().to_dict()}")
print(f"\nSize range: {tcga_brca_he['size_MB'].min()} – {tcga_brca_he['size_MB'].max()} MB")
tcga_brca_he.head(10)

TCGA-BRCA H&E slides: 3111
Tissue types: {'Neoplasm, Primary': 2704, 'Normal': 399}

Size range: 9.1 – 3586.3 MB


,SeriesInstanceUID,PatientID,ContainerIdentifier,tissue_type,width_px,height_px,pixel_spacing_mm,objective_power,size_MB
0,1.3.6.1.4.1.5962.99.1.1247079754.248369703.163...,TCGA-A7-A26J,TCGA-A7-A26J-01B-02-BS2,"Neoplasm, Primary",8618,7243,0.00025,40,9.1
1,1.3.6.1.4.1.5962.99.1.1279025007.575797294.163...,TCGA-AC-A2QJ,TCGA-AC-A2QJ-11A-02-TS2,Normal,33993,6923,0.00050,20,15.5
2,1.3.6.1.4.1.5962.99.1.1306193840.1223239355.16...,TCGA-AC-A2FB,TCGA-AC-A2FB-11A-01-TSA,Normal,31993,7690,0.00050,20,16.5
3,1.3.6.1.4.1.5962.99.1.1263739509.328647659.163...,TCGA-BH-A5IZ,TCGA-BH-A5IZ-11A-01-TS1,Normal,20041,15048,0.00050,20,18.7
4,1.3.6.1.4.1.5962.99.1.1290706954.176399971.163...,TCGA-AC-A2QJ,TCGA-AC-A2QJ-11A-01-TS1,Normal,41991,9316,0.00050,20,22.8
5,1.3.6.1.4.1.5962.99.1.1315471301.865934712.163...,TCGA-E9-A1RF,TCGA-E9-A1RF-11A-03-TSC,Normal,27888,9784,0.00050,20,23.0
6,1.3.6.1.4.1.5962.99.1.1322835661.1509329658.16...,TCGA-GM-A2D9,TCGA-GM-A2D9-11A-02-TS2,Normal,35992,8945,0.00050,20,25.0
7,1.3.6.1.4.1.5962.99.1.1255105493.445850269.163...,TCGA-E9-A1RF,TCGA-E9-A1RF-11A-01-TSA,Normal,23904,14929,0.00050,20,26.9
8,1.3.6.1.4.1.5962.99.1.1247939226.656219549.163...,TCGA-OL-A5RW,TCGA-OL-A5RW-01Z-00-DX1,"Neoplasm, Primary",12879,13618,0.00050,<NA>,27.1
9,1.3.6.1.4.1.5962.99.1.1244105102.1853648157.16...,TCGA-E9-A1NF,TCGA-E9-A1NF-11A-05-TSE,Normal,13943,17330,0.00049,20,27.3


### 1.4 Select a Representative Set of Slides

For the hands-on GrandQC run we select a small set of slides that covers:
- Different tissue types (tumor vs. normal)
- Different magnifications (20× and 40×)
- Manageable file sizes for a notebook environment

We cap the selection at slides under 200 MB to keep download and processing times reasonable.

In [6]:
# Pick up to 5 small slides with variety in tissue type and magnification
demo_slides = (
    tcga_brca_he[tcga_brca_he["size_MB"] < 200]
    .drop_duplicates(subset="tissue_type")
    .head(5)
    .reset_index(drop=True)
)

# Fall back to the 5 smallest slides if tissue_type is missing
if len(demo_slides) < 3:
    demo_slides = tcga_brca_he[tcga_brca_he["size_MB"] < 200].head(5).reset_index(drop=True)

print(f"Selected {len(demo_slides)} slides for GrandQC processing:")
display(
    demo_slides[[
        "PatientID", "ContainerIdentifier", "tissue_type",
        "objective_power", "width_px", "height_px", "size_MB"
    ]]
)

Selected 3 slides for GrandQC processing:


,PatientID,ContainerIdentifier,tissue_type,objective_power,width_px,height_px,size_MB
0,TCGA-A7-A26J,TCGA-A7-A26J-01B-02-BS2,"Neoplasm, Primary",40,8618,7243,9.1
1,TCGA-AC-A2QJ,TCGA-AC-A2QJ-11A-02-TS2,Normal,20,33993,6923,15.5
2,TCGA-E2-A15K,TCGA-E2-A15K-06A-01-TS1,None,40,33319,19962,85.7


In [7]:
# Preview slides in the IDC SLIM viewer (pathology viewer)
for _, row in demo_slides.iterrows():
    url = idc_client.get_viewer_URL(seriesInstanceUID=row["SeriesInstanceUID"])
    print(f"{row['ContainerIdentifier']}  ({row['tissue_type']}, {row['size_MB']} MB)")
    print(f"  {url}")

TCGA-A7-A26J-01B-02-BS2  (Neoplasm, Primary, 9.1 MB)
  https://viewer.imaging.datacommons.cancer.gov/slim/studies/2.25.313270449313106073714723178854675013710/series/1.3.6.1.4.1.5962.99.1.1247079754.248369703.1637629652298.2.0
TCGA-AC-A2QJ-11A-02-TS2  (Normal, 15.5 MB)
  https://viewer.imaging.datacommons.cancer.gov/slim/studies/2.25.337996839190358918582393097516172189888/series/1.3.6.1.4.1.5962.99.1.1279025007.575797294.1637661597551.2.0
TCGA-E2-A15K-06A-01-TS1  (None, 85.7 MB)
  https://viewer.imaging.datacommons.cancer.gov/slim/studies/2.25.54614374037792363265218330907954861388/series/1.3.6.1.4.1.5962.99.1.1231257225.1365336880.1637613829769.2.0


## Part 2: GrandQC Setup

Before running GrandQC we need to:
1. Clone the repository from GitHub
2. Install its Python dependencies
3. Download the pre-trained models from Zenodo
4. Apply a small patch so both scripts accept IDC's DICOM series directories alongside conventional WSI files

> **Note**: GrandQC's scripts currently iterate over files in a flat folder. IDC downloads each DICOM series into its own subdirectory. The patch below (≤ 10 lines changed across 2 files) teaches GrandQC to recognise a directory whose contents end in `.dcm` as a valid slide entry and resolves the OpenSlide path accordingly. No model weights or algorithm logic are changed.

### 2.1 Clone and Install Dependencies

In [8]:
%%capture
# Clone GrandQC (shallow clone — we only need the latest commit)
!git clone --depth 1 https://github.com/cpath-ukk/grandqc.git

# Install GrandQC Python dependencies.
# PyTorch is already present in Colab; we install the remaining packages.
# segmentation-models-pytorch 0.3.1 is pinned because its model.predict() API
# was removed in 0.4.0 and GrandQC relies on it.
!pip install -q \
    "segmentation-models-pytorch==0.3.1" \
    "rasterio>=1.3" \
    "imagecodecs" \
    "zarr<3" \
    "tifffile"

### 2.2 Download Pre-trained Models

In [9]:
import os

GRANDQC_SCRIPTS = "grandqc/01_WSI_inference_OPENSLIDE_QC"
MODELS_TD = os.path.join(GRANDQC_SCRIPTS, "models", "td")
MODELS_QC = os.path.join(GRANDQC_SCRIPTS, "models", "qc")
os.makedirs(MODELS_TD, exist_ok=True)
os.makedirs(MODELS_QC, exist_ok=True)

# Tissue detection model (~27 MB) — Zenodo record 14507273
!wget -q --show-progress -O {MODELS_TD}/Tissue_Detection_MPP10.pth \
    "https://zenodo.org/records/14507273/files/Tissue_Detection_MPP10.pth?download=1"

# Artifact segmentation model at MPP=1.5 (~25 MB) — Zenodo record 14041538
# MPP=1.5 (≈7× objective) is the recommended default for most scanners.
!wget -q --show-progress -O {MODELS_QC}/GrandQC_MPP15.pth \
    "https://zenodo.org/records/14041538/files/GrandQC_MPP15.pth?download=1"

print("Models downloaded:")
for f in [f"{MODELS_TD}/Tissue_Detection_MPP10.pth", f"{MODELS_QC}/GrandQC_MPP15.pth"]:
    size_mb = os.path.getsize(f) / 1024**2
    print(f"  {f}  ({size_mb:.1f} MB)")

grandqc/01_WSI_infe 100%[===================>]  25.41M  1.06MB/s    in 26s     
grandqc/01_WSI_infe 100%[===================>]  24.21M  1.08MB/s    in 25s     
Models downloaded:
  grandqc/01_WSI_inference_OPENSLIDE_QC/models/td/Tissue_Detection_MPP10.pth  (25.4 MB)
  grandqc/01_WSI_inference_OPENSLIDE_QC/models/qc/GrandQC_MPP15.pth  (24.2 MB)


### 2.3 Patch Scripts for DICOM Directory Support

IDC downloads each series as a directory of tiled DICOM files (`*.dcm`). OpenSlide 4.0+ can open any one of those files and reconstruct the full pyramid from the same directory.

The changes per script are:

| Script | Change |
|--------|--------|
| `wsi_tis_detect.py` | Replace file-only `os.listdir` filter with one that also yields subdirectories containing `.dcm` files; resolve the path before calling `OpenSlide()` |
| `main.py` | Same two changes using `open_slide()`; plus `torch.load(..., weights_only=False)` so the pickled QC model loads under PyTorch ≥ 2.6 |

Both scripts also get automatic CUDA/MPS/CPU device selection instead of the hardcoded `'cuda'`.

> **Note**: GrandQC's scripts iterate over files in a flat folder; IDC downloads each DICOM series into its own subdirectory, so the patch teaches them to recognise a directory whose contents end in `.dcm` as a valid slide entry and resolves the OpenSlide path accordingly. The `weights_only=False` change is required because PyTorch 2.6 made `weights_only=True` the default for `torch.load`, which refuses to unpickle the `segmentation_models_pytorch` model object that `main.py` loads. No model weights or algorithm logic are changed.

In [ ]:
from pathlib import Path

def patch_grandqc_for_dicom(script_path: str) -> None:
    text = Path(script_path).read_text()
    original = text

    # ── 1. Auto-detect compute device ──────────────────────────────────────
    text = text.replace(
        "DEVICE = 'cuda'",
        "import torch as _t; DEVICE = ('cuda' if _t.cuda.is_available() "
        "else ('mps' if _t.backends.mps.is_available() else 'cpu'))",
    )

    # ── 2. Replace slide-list builder to also accept DICOM directories ──────
    DICOM_ITER = (
        "def _iter_slides(d):\n"
        "    for n in sorted(os.listdir(d)):\n"
        "        p = os.path.join(d, n)\n"
        "        if os.path.isfile(p) or (\n"
        "                os.path.isdir(p) and\n"
        "                any(x.lower().endswith('.dcm') for x in os.listdir(p))):\n"
        "            yield n\n"
    )

    # wsi_tis_detect.py pattern
    OLD_LIST_TIS = (
        "slide_names = sorted([f for f in os.listdir(SLIDE_DIR) "
        "if os.path.isfile(os.path.join(SLIDE_DIR, f))])"
    )
    NEW_LIST_TIS = DICOM_ITER + "slide_names = list(_iter_slides(SLIDE_DIR))"

    # main.py pattern
    OLD_LIST_MAIN = "slide_names = sorted(os.listdir(SLIDE_DIR))"
    NEW_LIST_MAIN = DICOM_ITER + "slide_names = list(_iter_slides(SLIDE_DIR))"

    text = text.replace(OLD_LIST_TIS, NEW_LIST_TIS)
    text = text.replace(OLD_LIST_MAIN, NEW_LIST_MAIN)

    # ── 3. Resolve DICOM directory to first .dcm file before open ───────────
    RESOLVE = (
        "        if os.path.isdir(path_slide):\n"
        "            _dcms = sorted(x for x in os.listdir(path_slide)"
        " if x.lower().endswith('.dcm'))\n"
        "            path_slide = os.path.join(path_slide, _dcms[0])\n"
    )

    for open_call in ("slide = OpenSlide(path_slide)", "slide = open_slide(path_slide)"):
        old = f"        path_slide = os.path.join(SLIDE_DIR, slide_name)\n        {open_call}"
        new = f"        path_slide = os.path.join(SLIDE_DIR, slide_name)\n{RESOLVE}        {open_call}"
        text = text.replace(old, new)

    # ── 4. Load the full QC model with weights_only=False ───────────────────
    # main.py loads a pickled nn.Module via torch.load(). Since PyTorch 2.6 the
    # default weights_only=True refuses to unpickle the segmentation_models_pytorch
    # classes inside, raising UnpicklingError before the per-slide try/except and
    # killing the process with exit code 1. The Zenodo weights are trusted, so we
    # opt back into the full unpickling path. (wsi_tis_detect.py is unaffected —
    # it loads a plain state_dict, which weights_only=True allows.)
    text = text.replace(
        "model_prim = torch.load(MODEL_QC_DIR + MODEL_QC_NAME, map_location=DEVICE)",
        "model_prim = torch.load(MODEL_QC_DIR + MODEL_QC_NAME, map_location=DEVICE, "
        "weights_only=False)",
    )

    # ── 5. Progress reporting in main.py's slide loop ───────────────────────
    # Track failures and report [i/N] progress. (main.py-specific anchors — these
    # strings do not occur in wsi_tis_detect.py, so this is a no-op there.)
    text = text.replace(
        "for slide_name in slide_names[start:end]:\n"
        "    try:\n"
        "        # Register start time\n"
        "        start = timeit.default_timer()\n"
        "\n"
        "        print(\"\")\n"
        "        print(\"Processing:\", slide_name)",
        "_slides_to_run = slide_names[start:end]\n"
        "_failures = []\n"
        "print(f\"Found {len(_slides_to_run)} slide(s) to process.\")\n"
        "for _idx, slide_name in enumerate(_slides_to_run, 1):\n"
        "    try:\n"
        "        # Register start time\n"
        "        start = timeit.default_timer()\n"
        "\n"
        "        print(\"\")\n"
        "        print(f\"[{_idx}/{len(_slides_to_run)}] Processing:\", slide_name)",
    )

    # ── 6. Record per-slide failures and exit non-zero if any occurred ──────
    # The upstream loop swallows every per-slide error and the script exits 0,
    # so the notebook can't tell a slide failed. Collect failures and sys.exit(1)
    # after the loop so the caller surfaces them.
    text = text.replace(
        "    except Exception as e:\n"
        "        print(f\"There was some problem with the slide. The error is: {e}\")",
        "    except Exception as e:\n"
        "        print(f\"There was some problem with the slide. The error is: {e}\")\n"
        "        _failures.append((slide_name, str(e)))\n"
        "\n"
        "# Surface per-slide failures with a non-zero exit so the caller sees them\n"
        "if _failures:\n"
        "    print(f\"\\n{len(_failures)} of {len(_slides_to_run)} slide(s) FAILED:\")\n"
        "    for _name, _err in _failures:\n"
        "        print(f\"  - {_name}: {_err}\")\n"
        "    import sys as _sys; _sys.exit(1)\n"
        "print(f\"\\nAll {len(_slides_to_run)} slide(s) processed successfully.\")",
    )

    if text == original:
        print(f"  WARNING: no changes applied to {script_path}")
    else:
        Path(script_path).write_text(text)
        n = sum(1 for a, b in zip(original.splitlines(), text.splitlines()) if a != b)
        n += abs(len(text.splitlines()) - len(original.splitlines()))
        print(f"  Patched {script_path}  (+{n} lines changed)")


for script in [
    f"{GRANDQC_SCRIPTS}/wsi_tis_detect.py",
    f"{GRANDQC_SCRIPTS}/main.py",
]:
    patch_grandqc_for_dicom(script)
print("Done.")

In [ ]:
# Sanity-check: confirm the patches were inserted in both scripts
for script in [f"{GRANDQC_SCRIPTS}/wsi_tis_detect.py", f"{GRANDQC_SCRIPTS}/main.py"]:
    text = Path(script).read_text()
    has_dcm  = "_iter_slides" in text
    has_res  = "_dcms" in text
    has_dev  = "cuda.is_available" in text
    # weights_only=False only applies to main.py's full-model load
    has_wo   = ("weights_only=False" in text) if script.endswith("main.py") else "n/a"
    print(f"{script.split('/')[-1]:30s}  dicom_iter={has_dcm}  dcm_resolve={has_res}  "
          f"auto_device={has_dev}  weights_only_fix={has_wo}")

## Part 3: Download Slides from IDC

`idc_client.download_from_selection()` fetches DICOM files for the requested series and places each series in its own subdirectory named by `SeriesInstanceUID`.

After downloading we rename each subdirectory to its `ContainerIdentifier` (the original TCGA slide barcode). This ensures that GrandQC's output files and the pre-computed TCGA mask filenames are consistently keyed by the human-readable barcode rather than an opaque UID.

### 3.1 Download DICOM Series

In [ ]:
SLIDE_DIR = "slides"
os.makedirs(SLIDE_DIR, exist_ok=True)

print(f"Downloading {len(demo_slides)} slides  →  {SLIDE_DIR}/")
print("(This may take a few minutes depending on file sizes and network speed)\n")

idc_client.download_from_selection(
    downloadDir=SLIDE_DIR,
    seriesInstanceUID=demo_slides["SeriesInstanceUID"].tolist(),
    dirTemplate="%SeriesInstanceUID",
)
print("\nDownload complete.")

### 3.2 Rename Series Directories to TCGA Barcodes

GrandQC uses the directory/file name as the key for all output files (tissue mask, QC map, overlay, TSV report row). By renaming each downloaded series directory from its `SeriesInstanceUID` to its `ContainerIdentifier` (TCGA barcode), we get:

- Human-readable output filenames (`TCGA-A7-A26J-01B-02-BS2_MASK.png` instead of a UID)
- Automatic matching with GrandQC's pre-computed TCGA masks (Part 6), which are keyed by the same barcode

In [ ]:
# Build SeriesInstanceUID → ContainerIdentifier map
uid_to_barcode = dict(zip(demo_slides["SeriesInstanceUID"], demo_slides["ContainerIdentifier"]))

for uid, barcode in uid_to_barcode.items():
    src = os.path.join(SLIDE_DIR, uid)
    dst = os.path.join(SLIDE_DIR, barcode)
    if os.path.isdir(src):
        os.rename(src, dst)
        print(f"  {uid}  →  {barcode}")
    elif os.path.isdir(dst):
        print(f"  Already renamed: {barcode}")
    else:
        print(f"  WARNING: directory not found: {uid}")

print("\nSlide directories after renaming:")
for d in sorted(os.listdir(SLIDE_DIR)):
    dpath = os.path.join(SLIDE_DIR, d)
    if os.path.isdir(dpath):
        n = sum(1 for f in os.listdir(dpath) if f.lower().endswith(".dcm"))
        print(f"  {d}/  ({n} .dcm files)")

### 3.3 Verify OpenSlide Can Read the DICOM Files

OpenSlide 4.0 (bundled in `openslide-bin`) reads DICOM WSI files natively. Opening any one `.dcm` file from a series directory is sufficient — OpenSlide discovers the remaining pyramid levels from the other files in the same directory that share the same `SeriesInstanceUID`.

In [ ]:
import openslide

print(f"OpenSlide version: {openslide.__library_version__}\n")

for barcode in uid_to_barcode.values():
    slide_dir = os.path.join(SLIDE_DIR, barcode)
    dcm_files = sorted(f for f in os.listdir(slide_dir) if f.lower().endswith(".dcm"))
    if not dcm_files:
        print(f"  {barcode}: no .dcm files found")
        continue
    entry = os.path.join(slide_dir, dcm_files[0])
    try:
        slide = openslide.OpenSlide(entry)
        w, h = slide.level_dimensions[0]
        mpp = slide.properties.get("openslide.mpp-x", "N/A")
        vendor = slide.properties.get("openslide.vendor", "N/A")
        print(f"  {barcode}")
        print(f"    {w:,} × {h:,} px  |  MPP={mpp}  |  vendor={vendor}  |  levels={slide.level_count}")
        slide.close()
    except Exception as exc:
        print(f"  {barcode}: ERROR — {exc}")

print("\nAll slides verified.")

## Part 4: Run GrandQC Quality Control

GrandQC runs as two sequential scripts, both inside `grandqc/01_WSI_inference_OPENSLIDE_QC/`:

| Step | Script | MPP | Time (GPU) | Output |
|------|--------|-----|------------|--------|
| 1 | `wsi_tis_detect.py` | 10 | ~0.4 s/slide | Binary tissue masks |
| 2 | `main.py` | 1.5 | ~30–45 s/slide | 7-class artifact maps, GeoJSON, TSV report |

The 7 artifact classes are: **1** clean tissue · **2** folds · **3** dark spots · **4** pen marks · **5** air bubbles/edges · **6** out-of-focus · **7** background.

Both scripts load models via relative paths (`./models/…`) and must therefore be invoked with their directory as the working directory. We use `subprocess` with `cwd=GRANDQC_SCRIPTS` to achieve this while keeping output streaming to the cell.

> **On inference resolution**: the artifact model runs at MPP 1.5 (≈7× objective), *not* at the slide's native 20×/40×, and the per-slide MPP is read from OpenSlide's `openslide.mpp-x` property (derived from the DICOM `PixelSpacing` for IDC slides). For how this is controlled, the patch-size math, and an empirical confirmation on an IDC slide, see [`grandqc_slide_quality_resolution_notes.md`](grandqc_slide_quality_resolution_notes.md).

### 4.1 Step 1 — Tissue Detection

Produces a low-resolution binary mask (tissue vs. background) for each slide. The mask is saved as a PNG and re-used by Step 2 to skip background patches and cut inference time.

In [ ]:
import subprocess, sys

OUTPUT_DIR = "qc_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

slide_dir_abs  = os.path.abspath(SLIDE_DIR)
output_dir_abs = os.path.abspath(OUTPUT_DIR)
scripts_abs    = os.path.abspath(GRANDQC_SCRIPTS)

print(f"Slides:          {slide_dir_abs}")
print(f"Output:          {output_dir_abs}")
print(f"Scripts dir:     {scripts_abs}")
print()
print("Running wsi_tis_detect.py ...")

result = subprocess.run(
    [sys.executable, "wsi_tis_detect.py",
     "--slide_folder", slide_dir_abs,
     "--output_dir",   output_dir_abs],
    cwd=scripts_abs,
)

if result.returncode != 0:
    print(f"\nERROR: wsi_tis_detect.py exited with code {result.returncode}")
else:
    mask_dir = os.path.join(output_dir_abs, "tis_det_mask")
    masks = sorted(os.listdir(mask_dir))
    print(f"\nTissue detection complete — {len(masks)} mask(s) written to {mask_dir}/")
    for m in masks:
        print(f"  {m}")

### 4.2 Step 2 — Artifact Segmentation

Runs the 7-class artifact model at MPP=1.5. For each slide it produces:

- `maps_qc/<barcode>_map_QC.png` — color-coded artifact map (same resolution as tissue mask)
- `overlays_qc/<barcode>_overlay_QC.jpg` — artifact map overlaid on a downscaled slide thumbnail
- `mask_qc/<barcode>_mask.png` — raw integer mask (class 1–7 per pixel)
- `geojson_qc/<barcode>.geojson` — polygon annotations compatible with QuPath and SLIM
- `report_qc_output_0_N_stats_per_slide.txt` — TSV with per-slide dimensions and processing time

> **GPU note**: On a T4 GPU (Colab free tier), expect ~30–45 s per slide. On CPU only, plan for 10–30 min per slide depending on size.

In [ ]:
print("Running main.py (artifact segmentation) ...")

result = subprocess.run(
    [sys.executable, "main.py",
     "--slide_folder", slide_dir_abs,
     "--output_dir",   output_dir_abs,
     "--mpp_model",    "1.5",
     "--create_geojson", "Y"],
    cwd=scripts_abs,
)

if result.returncode != 0:
    print(f"\nERROR: main.py exited with code {result.returncode}")
else:
    maps_dir = os.path.join(output_dir_abs, "maps_qc")
    maps = sorted(os.listdir(maps_dir))
    print(f"\nArtifact segmentation complete — {len(maps)} map(s) in {maps_dir}/")
    for m in maps:
        print(f"  {m}")

    # Locate the TSV report (name includes the slide count)
    reports = [f for f in os.listdir(output_dir_abs) if f.endswith("_stats_per_slide.txt")]
    if reports:
        import pandas as pd
        report_path = os.path.join(output_dir_abs, reports[0])
        df = pd.read_csv(report_path, sep="\t")
        print(f"\nPer-slide report ({reports[0]}):")
        display(df[["slide_name", "mpp", "height", "width", "time"]])

## Part 5: Visualize and Analyze Results

GrandQC writes several output types per slide. We look at three of them:

1. **Tissue detection overlays** — confirms that tissue vs. background separation was correct
2. **Artifact segmentation maps** — color-coded composite of the 7-class artifact model overlaid on a slide thumbnail
3. **Raw integer masks** (`mask_qc/`) — pixel-level class labels (1–7) used to compute per-slide artifact fractions

### 5.1 Tissue Detection Overlays

Each overlay blends the original thumbnail (70 %) with a two-class color mask (30 %): blue = tissue, gray = background.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

overlay_dir = os.path.join(OUTPUT_DIR, "tis_det_overlay")
fnames = sorted(os.listdir(overlay_dir))

fig, axes = plt.subplots(1, len(fnames), figsize=(6 * len(fnames), 5))
if len(fnames) == 1:
    axes = [axes]
for ax, fname in zip(axes, fnames):
    ax.imshow(mpimg.imread(os.path.join(overlay_dir, fname)))
    ax.set_title(fname.replace("_OVERLAY.jpg", ""), fontsize=7)
    ax.axis("off")
plt.suptitle("Step 1 — Tissue Detection", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### 5.2 Artifact Segmentation Maps

The QC overlay composites the artifact color map onto a downscaled thumbnail of the original slide. Colors match the `wsi_colors.py` palette used by GrandQC:

| Class | Color | Artifact |
|-------|-------|---------|
| 1 | ■ Gray `#808080` | Clean tissue |
| 2 | ■ Tomato `#FF6347` | Folds |
| 3 | ■ Lime `#00FF00` | Dark spots |
| 4 | ■ Red `#FF0000` | Pen marks |
| 5 | ■ Magenta `#FF00FF` | Air bubbles / slide edges |
| 6 | ■ Indigo `#4B0082` | Out-of-focus |
| 7 | □ White `#FFFFFF` | Background |

In [ ]:
overlay_qc_dir = os.path.join(OUTPUT_DIR, "overlays_qc")
fnames_qc = sorted(os.listdir(overlay_qc_dir))

fig, axes = plt.subplots(1, len(fnames_qc), figsize=(6 * len(fnames_qc), 5))
if len(fnames_qc) == 1:
    axes = [axes]
for ax, fname in zip(axes, fnames_qc):
    ax.imshow(mpimg.imread(os.path.join(overlay_qc_dir, fname)))
    ax.set_title(fname.replace("_overlay_QC.jpg", ""), fontsize=7)
    ax.axis("off")
plt.suptitle("Step 2 — Artifact Segmentation Overlays", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

### 5.3 Artifact Fractions per Slide

We read the raw integer masks from `mask_qc/` (pixel values 1–7) and compute what fraction of the **tissue area** (excluding background, class 7) belongs to each artifact class. This is the standard metric reported in the GrandQC paper.

In [ ]:
from PIL import Image

# GrandQC wsi_colors.py palette — RGB, indices 0-5 correspond to classes 1-6
GRANDQC_COLORS_RGB = [
    [128, 128, 128],  # 1 Clean tissue
    [255,  99,  71],  # 2 Folds
    [  0, 255,   0],  # 3 Dark spots
    [255,   0,   0],  # 4 Pen marks
    [255,   0, 255],  # 5 Bubbles/edges
    [ 75,   0, 130],  # 6 Out-of-focus
]
CLASS_NAMES_SHORT = [
    "Clean tissue", "Folds", "Dark spots",
    "Pen marks", "Bubbles/edges", "Out-of-focus",
]
# Normalize to [0,1] for matplotlib
mpl_colors = [[r/255, g/255, b/255] for r, g, b in GRANDQC_COLORS_RGB]

mask_dir = os.path.join(OUTPUT_DIR, "mask_qc")
rows = []
for fname in sorted(os.listdir(mask_dir)):
    barcode = fname.replace("_mask.png", "")
    mask = np.array(Image.open(os.path.join(mask_dir, fname)))
    tissue_px = np.sum(mask != 7)  # exclude background
    if tissue_px == 0:
        continue
    row = {"slide": barcode}
    for cls_idx, name in enumerate(CLASS_NAMES_SHORT, start=1):
        row[name] = np.sum(mask == cls_idx) / tissue_px * 100
    rows.append(row)

stats_df = pd.DataFrame(rows).set_index("slide")
print("Artifact class fractions (% of tissue area):")
display(stats_df.round(1))

# ── Stacked bar chart ────────────────────────────────────────────────────────
ax = stats_df[CLASS_NAMES_SHORT].plot(
    kind="bar",
    stacked=True,
    color=mpl_colors,
    figsize=(max(6, 2.5 * len(stats_df)), 5),
    edgecolor="none",
)
ax.set_ylabel("% of tissue area")
ax.set_xlabel("")
ax.set_title("GrandQC Artifact Fractions per Slide", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right", fontsize=8)
ax.legend(loc="upper right", fontsize=8, framealpha=0.8)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

### 5.4 View Original Slides in IDC SLIM Viewer

The IDC SLIM viewer is a web-based pathology viewer for DICOM whole slide images. The links below open each slide at full resolution. The GeoJSON annotations produced by GrandQC (`geojson_qc/`) can be loaded into QuPath or other tools that accept GeoJSON overlays for interactive artifact review.

In [ ]:
from IPython.display import IFrame, display

print("IDC SLIM viewer links (open in browser for full-resolution viewing):\n")
for _, row in demo_slides.iterrows():
    url = idc_client.get_viewer_URL(seriesInstanceUID=row["SeriesInstanceUID"])
    print(f"  {row['ContainerIdentifier']}")
    print(f"    {url}\n")

# Embed the first slide inline (750 px height)
first_url = idc_client.get_viewer_URL(seriesInstanceUID=demo_slides.iloc[0]["SeriesInstanceUID"])
print(f"Embedding: {demo_slides.iloc[0]['ContainerIdentifier']}")
display(IFrame(first_url, width="100%", height=750))

## Part 6: Validating the Pipeline Against Pre-computed TCGA Masks

In Parts 3–5 we ran GrandQC locally on IDC DICOM slides using OpenSlide's native DICOM reader — a path the GrandQC authors did not test in their original publication. Before using the pipeline on new data, it is good practice to verify that our results are consistent with the authors' reference outputs.

The GrandQC authors published pre-computed QC masks for all 32 TCGA cohorts on Zenodo:

> **GrandQC pre-computed masks**: [zenodo.org/records/14041578](https://zenodo.org/records/14041578)  
> 36 tar archives — one per TCGA cohort (some large cohorts split into two parts) — totalling ~18.3 GB.

Each archive unpacks to PNG files named `{ContainerIdentifier}_mask.png` with integer labels 1–7 — the same format and naming convention produced by `main.py`. By downloading the TCGA-BRCA archive and comparing artifact fractions for our demo slides against the reference, we can confirm that the DICOM-based pipeline is producing equivalent results.

### 6.1 Available Cohort Archives

The Zenodo record contains one tar file per TCGA cohort. Large cohorts (BRCA, THCA, GBM, LUAD, UCEC, LGG, TGCT) are split into two parts.

In [ ]:
import requests

# Fetch the Zenodo record metadata to get the authoritative file list
rec = requests.get("https://zenodo.org/api/records/14041578").json()
files = rec["files"]

rows_z = []
for f in sorted(files, key=lambda x: x["key"]):
    rows_z.append({
        "archive": f["key"],
        "size_MB": round(f["size"] / 1024**2, 0),
    })

zenodo_df = pd.DataFrame(rows_z)
total_gb  = zenodo_df["size_MB"].sum() / 1024
print(f"{len(zenodo_df)} archives  |  total: {total_gb:.1f} GB\n")
display(zenodo_df)

### 6.2 Download the TCGA-BRCA Reference Masks

`BRCA.tar` (~2 GB) contains masks for all TCGA-BRCA slides. After extraction, each file is named `{ContainerIdentifier}_mask.png` — identical to what GrandQC writes into `mask_qc/` locally.

In [ ]:
REF_MASKS_DIR = "brca_ref_masks"
os.makedirs(REF_MASKS_DIR, exist_ok=True)

# Download BRCA.tar (~2 GB) — takes 5-10 min on Colab
!wget -q --show-progress \
    "https://zenodo.org/records/14041578/files/BRCA.tar?download=1" \
    -O BRCA.tar

# Extract — PNG files land directly in the archive root
!tar xf BRCA.tar -C {REF_MASKS_DIR}

# Report what we got
ref_masks = sorted(f for f in os.listdir(REF_MASKS_DIR) if f.endswith("_mask.png"))
print(f"\nExtracted {len(ref_masks)} reference masks to {REF_MASKS_DIR}/")
print("Sample filenames:")
for f in ref_masks[:5]:
    print(f"  {f}")

### 6.3 Pipeline Validation: Compare Local vs. Reference Masks

The reference masks in `brca_ref_masks/` were produced by the GrandQC authors running on TIFF/SVS files. Our local masks in `qc_output/mask_qc/` were produced by the same model running on IDC DICOM files via OpenSlide's native DICOM reader.

We compare per-class artifact fractions for each demo slide. Close agreement (< 2 percentage points per class) confirms that the DICOM-based path does not introduce bias relative to the authors' original pipeline.

In [ ]:
def mask_fractions(mask_path):
    mask = np.array(Image.open(mask_path))
    tissue_px = np.sum(mask != 7)
    if tissue_px == 0:
        return None
    return {name: np.sum(mask == cls) / tissue_px * 100
            for cls, name in enumerate(CLASS_NAMES_SHORT, start=1)}


# Build comparison table for each demo slide
compare_rows = []
for barcode in uid_to_barcode.values():
    local_path = os.path.join(OUTPUT_DIR, "mask_qc", f"{barcode}_mask.png")
    ref_path   = os.path.join(REF_MASKS_DIR, f"{barcode}_mask.png")

    if not os.path.exists(local_path):
        print(f"  MISSING local mask: {barcode}")
        continue
    if not os.path.exists(ref_path):
        print(f"  MISSING reference mask: {barcode} — slide may not be in BRCA.tar")
        continue

    local_frac = mask_fractions(local_path)
    ref_frac   = mask_fractions(ref_path)
    if local_frac is None or ref_frac is None:
        continue

    for name in CLASS_NAMES_SHORT:
        compare_rows.append({
            "slide":    barcode,
            "class":    name,
            "local_%":  round(local_frac[name], 2),
            "ref_%":    round(ref_frac[name], 2),
            "delta_pp": round(local_frac[name] - ref_frac[name], 2),
        })

compare_df = pd.DataFrame(compare_rows)
print("Per-class fraction comparison (local DICOM run vs. Zenodo reference):\n")
display(compare_df.pivot_table(index="slide", columns="class", values="delta_pp").round(2))
print("\nδ = local − reference (percentage points).  Values near 0 confirm agreement.")

In [ ]:
# Scatter plot: local fraction vs. reference fraction for all classes × slides
fig, ax = plt.subplots(figsize=(5, 5))

class_colors = {name: mpl_colors[i] for i, name in enumerate(CLASS_NAMES_SHORT)}

for cls_name, grp in compare_df.groupby("class"):
    ax.scatter(grp["ref_%"], grp["local_%"],
               label=cls_name, color=class_colors[cls_name],
               s=60, edgecolors="black", linewidths=0.4, zorder=3)

# Identity line
lim_max = max(compare_df[["local_%", "ref_%"]].max()) * 1.05
ax.plot([0, lim_max], [0, lim_max], "k--", linewidth=0.8, label="y = x")
ax.set_xlabel("Reference fraction (%) — Zenodo")
ax.set_ylabel("Local fraction (%) — DICOM run")
ax.set_title("Pipeline Validation: Local vs. Reference Artifact Fractions",
             fontweight="bold", fontsize=9)
ax.legend(fontsize=7, framealpha=0.8)
ax.set_xlim(0, lim_max)
ax.set_ylim(0, lim_max)
plt.tight_layout()
plt.show()

### 6.4 Cohort-level QC Statistics

With validation confirmed, we compute per-slide artifact fractions across all TCGA-BRCA slides in the reference archive. This gives a population-level view of slide quality without running any inference locally — each mask is read once and discarded, keeping memory usage constant.

In [ ]:
from tqdm.notebook import tqdm

cohort_rows = []
for fname in tqdm(ref_masks, desc="Computing stats"):
    barcode = fname.replace("_mask.png", "")
    fracs = mask_fractions(os.path.join(REF_MASKS_DIR, fname))
    if fracs is not None:
        fracs["slide"] = barcode
        cohort_rows.append(fracs)

cohort_df = pd.DataFrame(cohort_rows).set_index("slide")

print(f"Cohort: TCGA-BRCA  |  {len(cohort_df)} slides\n")
print("Median artifact fractions (% of tissue area):")
display(cohort_df[CLASS_NAMES_SHORT].median().rename("median_%").to_frame().T.round(1))
print()
print("90th-percentile artifact fractions:")
display(cohort_df[CLASS_NAMES_SHORT].quantile(0.9).rename("p90_%").to_frame().T.round(1))

In [ ]:
# Violin plot of per-slide artifact fractions across TCGA-BRCA
# Exclude Clean tissue (class 1) — typically >80%, dominates the y-axis
artifact_classes = [c for c in CLASS_NAMES_SHORT if c != "Clean tissue"]
artifact_colors  = [mpl_colors[i] for i, c in enumerate(CLASS_NAMES_SHORT) if c != "Clean tissue"]

fig, ax = plt.subplots(figsize=(9, 4))
data    = [cohort_df[cls].dropna().values for cls in artifact_classes]
parts   = ax.violinplot(data, positions=range(len(artifact_classes)),
                        showmedians=True, showextrema=False)

for pc, color in zip(parts["bodies"], artifact_colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.75)
parts["cmedians"].set_color("black")
parts["cmedians"].set_linewidth(1.5)

ax.set_xticks(range(len(artifact_classes)))
ax.set_xticklabels(artifact_classes, rotation=20, ha="right")
ax.set_ylabel("% of tissue area")
ax.set_title(f"Artifact fraction distribution — TCGA-BRCA ({len(cohort_df)} slides)",
             fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Join QC stats with IDC sm_index metadata via ContainerIdentifier
cohort_meta = cohort_df.reset_index().rename(columns={"slide": "ContainerIdentifier"})
brca_meta   = tcga_brca_he[["ContainerIdentifier", "tissue_type"]].dropna(subset=["tissue_type"])

merged = cohort_meta.merge(brca_meta, on="ContainerIdentifier", how="inner")

print(f"Slides with tissue-type annotation: {len(merged)} / {len(cohort_df)}\n")

# Median artifact fractions by tissue type
by_tissue = (
    merged.groupby("tissue_type")[CLASS_NAMES_SHORT]
    .median()
    .round(1)
)
print("Median artifact fractions (%) by tissue type:")
display(by_tissue)